# F6 · Steering a frozen model with ten switches: a gentle introduction to elite search
**Series F: Quantum Meets Machine Learning, Post 6** &nbsp;|&nbsp; tutorial notebook for the post of the same name

**The story.** A *virtual cell* is a machine-learning model that predicts how a population of cells responds when one gene is turned down. Its predictions are judged by a score that cannot be differentiated, so the model cannot be tuned by gradient descent. A recent study (Yu and Batista, 2026) therefore **freezes** a trained model, replaces the ten numbers it uses internally by **ten binary switches** ($2^{10}=1024$ settings), and finds a good setting by an **elite search** (the cross-entropy method) that uses only the score. This notebook rebuilds, step by step and with plain NumPy, everything in the post: a six-bit example you can check by hand, the search and its limits, a test where the answer is known, how to judge a search honestly (baselines, cost, noise, the *winner's curse*), and why predicting the best setting *without* searching is a different problem.

> **You need** only high-school probability. Every term is explained when it first appears (the post has a glossary).
>
> **Two parts.** **Part I** needs NumPy, pandas and matplotlib and runs in about a minute on a CPU. **Part II** is optional (SciPy, and the public model code).
>
> **How to use it.** Each section starts with a short **Stop and predict** question. Answer it on paper before you run the cell.

| Section | What you do |
|---|---|
| 0 | cells as points, and the scale of the score |
| 1 | the elite search by hand: a six-bit worked example |
| 2 | why half a pseudocount: the ceiling of the search |
| 3 | a test with a known answer (24 bits) |
| 4 | judging a search: the work it does |
| 5 | judging a search: noise and the winner's curse |
| 6 | the result on real cells (as reported) |
| 7 | can the search be skipped? the actor, and the birthday problem |

## Helper functions
One cell with the functions and the figure code used below (read it when you are curious; you do not need it to follow the text).

In [ ]:
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
sigm = lambda x: 1 / (1 + np.exp(-x)); logit = lambda p: np.log(p / (1 - p))
alpha_t = lambda t: 0.40 / (1 + 0.015 * t) ** 0.20            # the step size schedule: 0.40 at the start, slowly falling
def entropy_bits(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12); return float(np.mean(-(p * np.log2(p) + (1 - p) * np.log2(1 - p))))
def polarization(p): return float(np.mean(2 * np.abs(np.asarray(p, float) - 0.5)))
def cem(reward, d, N=1024, K=102, T=80, seed=0, fresh=1024, opt=None, step=alpha_t, keep_p=False):
    """Bernoulli elite search (a discrete cross-entropy method): sample N codes, score them, keep the best K, move the bit probabilities toward the elite frequencies"""
    rg = np.random.default_rng(seed); th = np.zeros(d); hist = []; P = []; seen = set()
    for t in range(T):
        p = sigm(th); B = (rg.random((N, d)) < p).astype(int); r = reward(B, rg); seen.update(map(tuple, B))
        el = np.argsort(-r, kind="stable")[:K]; q = (B[el].sum(0) + 0.5) / (K + 1); a = step(t); th = (1 - a) * th + a * logit(q)
        p = sigm(th); Fr = (rg.random((fresh, d)) < p).astype(int); rf = reward(Fr, rg); P.append(p)
        hist.append(dict(cycle=t, mean=rf.mean(), best=rf.max(), H=entropy_bits(p), P=polarization(p), acc=float(np.mean((p > 0.5) == (opt == 1))) if opt is not None else np.nan, uniq=len(seen)))
    return (pd.DataFrame(hist), th, np.array(P)) if keep_p else (pd.DataFrame(hist), th)
# ---------- the worked example: 6 bits, a hidden target, 12 samples, 3 elites, a fixed step 0.4 ----------
TARGET6 = np.array([1, 0, 1, 1, 0, 1])
def worked_example(seed=11, N=12, K=3, a=0.4):
    rg = np.random.default_rng(seed); th = np.zeros(6); p = sigm(th); B = (rg.random((N, 6)) < p).astype(int); r = 1 - np.abs(B - TARGET6).sum(1) / 6
    order = np.argsort(-r, kind="stable"); el = order[:K]; c = B[el].sum(0); q = (c + 0.5) / (K + 1); lam = logit(q); th_new = (1 - a) * th + a * lam
    return dict(B=B, r=r, order=order, el=el, c=c, q=q, lam=lam, th_new=th_new, p_new=sigm(th_new), p_old=p)
# ---------- figures ----------
def fig_cells(seed=1):
    rg = np.random.default_rng(seed); ctrl = rg.normal([0, 0], [1.0, 0.7], (150, 2)); real = rg.normal([-1.6, -0.9], [0.9, 0.7], (150, 2)); pred = rg.normal([-1.3, -0.6], [0.9, 0.7], (150, 2)); bad = rg.normal([0.1, 0.0], [1.0, 0.7], (150, 2))
    fig, ax = plt.subplots(1, 3, figsize=(10.6, 3.3), sharex=True, sharey=True)
    ax[0].scatter(*ctrl.T, s=8, color=COLORS["gray"], alpha=0.7); ax[0].set_title("(a) control cells (gene g untouched)", fontsize=9)
    ax[1].scatter(*ctrl.T, s=8, color=COLORS["gray"], alpha=0.25); ax[1].scatter(*real.T, s=8, color=COLORS["blue"], alpha=0.8); ax[1].set_title("(b) measured: gene g knocked down", fontsize=9)
    ax[2].scatter(*real.T, s=8, color=COLORS["blue"], alpha=0.25); ax[2].scatter(*pred.T, s=8, color=COLORS["orange"], alpha=0.8); ax[2].set_title("(c) predicted by a model (orange)", fontsize=9)
    for a_ in ax: a_.set_xlabel("expression of gene 1"); a_.set_xticks([]); a_.set_yticks([])
    ax[0].set_ylabel("expression of gene 2"); plt.tight_layout(); return fig
def fig_scale():
    fig, ax = plt.subplots(2, 1, figsize=(9.5, 4.2), gridspec_kw=dict(height_ratios=[1, 1.15], hspace=0.9))
    a = ax[0]; a.axhline(0, color="k", lw=1); a.set_xlim(-0.1, 1.15); a.set_ylim(-1, 1); a.axis("off"); a.axvspan(0.15, 0.25, ymin=0.4, ymax=0.6, color=COLORS["orange"], alpha=0.5)
    for x, lab, col in ((0, "0 = baseline\n(predict the average perturbation response)", COLORS["gray"]), (1, "1 = replicate\n(an identical repeat of the experiment)", COLORS["green"])): a.plot([x], [0], "o", ms=11, color=col); a.text(x, -0.3, lab, ha="center", va="top", fontsize=8.5)
    a.text(0.2, 0.35, "the three scores of the\nstudy lie here (zoom below)", ha="center", fontsize=8.5, color=COLORS["orange"]); a.annotate("", xy=(0.95, 0.12), xytext=(0.3, 0.12), arrowprops=dict(arrowstyle="->", color=COLORS["gray"])); a.text(0.62, 0.3, "better predictions", ha="center", fontsize=9, color=COLORS["gray"])
    b = ax[1]; b.axhline(0, color="k", lw=1); b.set_xlim(0.185, 0.235); b.set_ylim(-1.4, 1.2); b.axis("off")
    for x, lab, col, up in ((0.1914, "default code\n0.1914", COLORS["blue"], True), (0.2233, "found by the search\n0.2233", COLORS["orange"], True), (0.2288, "best of all 1024 codes\n0.2288", COLORS["red"], False)):
        b.plot([x], [0], "o", ms=9, color=col)
        if up: b.plot([x, x], [0, 0.45], color=col, lw=1); b.text(x, 0.55, lab, ha="center", fontsize=8.5, color=col)
        else: b.plot([x, x], [0, -0.55], color=col, lw=1); b.text(x + 0.0005, -1.0, lab, ha="center", fontsize=8.5, color=col)
    for x in (0.19, 0.20, 0.21, 0.22, 0.23): b.plot([x, x], [-0.08, 0.08], color="k", lw=0.8); b.text(x, 0.2 if x > 0.2 else -0.25, f"{x:.2f}", ha="center", fontsize=8, color="gray")
    return fig
def fig_cem_heat(seed=4):
    d = 8; opt = np.array([1, 0, 1, 1, 0, 0, 1, 0]); rew = lambda B, rg: 1 - np.abs(B - opt).sum(1) / d
    h, th, P = cem(rew, d, N=60, K=6, T=14, seed=seed, fresh=60, opt=opt, keep_p=True, step=lambda t: 0.4)
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw=dict(width_ratios=[1.3, 1])); im = ax[0].imshow(np.vstack([np.full(d, 0.5), P]), cmap="coolwarm", vmin=0, vmax=1, aspect="auto"); ax[0].set(xlabel="bit", ylabel="cycle (0 = start)", title="(a) probability that each bit is 1"); ax[0].set_xticks(range(d)); ax[0].set_xticklabels([f"{i + 1}\n(target {opt[i]})" for i in range(d)], fontsize=7); plt.colorbar(im, ax=ax[0], fraction=0.04)
    ax[1].plot(range(1, 15), h["mean"], "o-", color=COLORS["blue"], label="average score of a fresh sample"); ax[1].plot(range(1, 15), h["H"], "s-", color=COLORS["orange"], label="uncertainty (entropy)"); ax[1].set(xlabel="cycle", title="(b) the search sharpens", ylim=(0, 1.05)); ax[1].legend(frameon=False, fontsize=7); plt.tight_layout(); return fig
def fig_pseudocount():
    fig, ax = plt.subplots(1, 3, figsize=(11.5, 3.5)); c = np.arange(0, 11)
    ax[0].plot(c / 10, c / 10, "--", color=COLORS["gray"], label="raw frequency c/K"); ax[0].plot(c / 10, (c + 0.5) / 11, "o-", color=COLORS["blue"], label="with the 1/2 pseudocounts"); ax[0].set(xlabel="fraction of the K = 10 elites with bit = 1", ylabel="value used by the update", title="(a) never exactly 0 or 1"); ax[0].legend(frameon=False, fontsize=7)
    x = np.linspace(0.01, 0.99, 99); ax[1].plot(x, logit(x), color=COLORS["orange"]); ax[1].set(xlabel="probability p", ylabel="log-odds  log(p / (1-p))", title="(b) log-odds stretch the ends", ylim=(-5, 5)); ax[1].axhline(0, color="k", lw=0.5)
    for K, col in ((10, COLORS["green"]), (26, COLORS["orange"]), (102, COLORS["blue"])):
        th = 0.0; ps = []
        for t in range(40): q = (K + 0.5) / (K + 1); th = (1 - alpha_t(t)) * th + alpha_t(t) * logit(q); ps.append(sigm(th))
        ax[2].plot(range(1, 41), ps, color=col, label=f"K = {K}: ceiling {(K + 0.5) / (K + 1):.4f}")
    ax[2].set(xlabel="cycle", ylabel="probability of the winning bit value", title="(c) a bit that is always in the elite", ylim=(0.5, 1.01)); ax[2].legend(frameon=False, fontsize=7); plt.tight_layout(); return fig
REPORTED = pd.DataFrame({"cycle": [0, 4, 7, 10, 20, 79], "mean": [0.5284, 0.8448, 0.9816, 0.9923, 0.9942, 0.9940], "H": [0.9826, 0.5892, 0.1202, 0.0549, 0.0444, 0.0443], "P": [0.1336, 0.6657, 0.9651, 0.9875, 0.9903, 0.9903]}).set_index("cycle")
def run_synthetic(seed=0):
    d = 24; opt = np.random.default_rng(1).integers(0, 2, d); rew = lambda B, rg: 1 - np.abs(B - opt).sum(1) / d; h, th = cem(rew, d, opt=opt, seed=seed); return h, opt
def fig_synthetic(h=None):
    if h is None: h, _ = run_synthetic()
    fig, ax = plt.subplots(1, 3, figsize=(11.5, 3.4)); pairs = (("mean", "average score of fresh samples", COLORS["blue"]), ("H", "uncertainty H (bits per bit)", COLORS["orange"]), ("P", "polarization P (decisiveness)", COLORS["green"]))
    for a_, (k, lab, col) in zip(ax, pairs): a_.plot(h.cycle, h[k], color=col, label="our run"); a_.scatter(REPORTED.index, REPORTED[k], marker="s", s=26, color="k", zorder=3, label="values reported by the study"); a_.set(xlabel="cycle", title=lab, xlim=(-2, 80))
    ax[1].axhline(0.0443, color="gray", ls=":", lw=0.9); ax[2].axhline(0.9903, color="gray", ls=":", lw=0.9); ax[0].legend(frameon=False, fontsize=7, loc="lower right"); plt.tight_layout(); return fig
d10 = 10; codes10 = ((np.arange(1 << d10)[:, None] >> np.arange(d10)) & 1) * 2 - 1; idx10 = lambda B: (B * (1 << np.arange(d10))).sum(1)
def landscape(kind, rg):
    if kind == "additive": true = codes10 @ rg.normal(size=d10)
    elif kind == "pairwise": true = codes10 @ (0.6 * rg.normal(size=d10)) + np.einsum("ni,ij,nj->n", codes10, np.triu(0.5 * rg.normal(size=(d10, d10)), 1), codes10)
    else: true = rg.normal(size=1 << d10)
    return (true - true.mean()) / true.std()
def compare(kind, sigma, reps=20):
    gaps = {k: [] for k in ("enumeration", "N=1024", "N=256", "N=64")}; uniq = {k: [] for k in ("N=1024", "N=256", "N=64")}
    for r in range(reps):
        rg = np.random.default_rng(1000 + r); true = landscape(kind, rg); best = true.max(); noisy = lambda B, g: true[idx10(B)] + sigma * g.normal(size=len(B))
        gaps["enumeration"].append(best - true[np.argmax(true + sigma * rg.normal(size=1 << d10))])
        for name, (N, K) in {"N=1024": (1024, 102), "N=256": (256, 26), "N=64": (64, 6)}.items():
            h, th = cem(noisy, d10, N=N, K=K, T=20, seed=r, fresh=8); code = (sigm(th) > 0.5).astype(int)[None]; gaps[name].append(best - true[idx10(code)[0]]); uniq[name].append(h.uniq.iloc[-1])
    return {k: float(np.mean(v)) for k, v in gaps.items()}, {k: float(np.mean(v)) for k, v in uniq.items()}
def budget_table():
    rows = []
    for kind in ("additive", "pairwise", "unstructured"):
        for sg in (0.0, 1.0): g, u = compare(kind, sg); rows.append((kind, sg, g["enumeration"], g["N=1024"], g["N=256"], g["N=64"], u["N=1024"], u["N=256"], u["N=64"]))
    return pd.DataFrame(rows, columns=["landscape", "noise", "gap enum", "gap N=1024", "gap N=256", "gap N=64", "codes N=1024", "codes N=256", "codes N=64"])
def fig_budget(B):
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); add = B[B.landscape == "additive"]; x = np.arange(2)
    for k, (col, lab) in enumerate((("codes N=1024", "N = 1024 per cycle"), ("codes N=256", "N = 256"), ("codes N=64", "N = 64"))): ax[0].bar(x + (k - 1) * 0.22, add[col], 0.22, color=(COLORS["red"], COLORS["orange"], COLORS["green"])[k], label=lab)
    ax[0].axhline(1024, color="k", ls="--", lw=0.9); ax[0].text(1.45, 1040, "all 1024 codes", ha="right", fontsize=8); ax[0].set(xticks=x, xticklabels=["exact scores", "noisy scores"], ylabel="different codes scored in 20 cycles", title="(a) what the search costs", ylim=(0, 1350)); ax[0].legend(frameon=False, fontsize=7, loc="upper left", ncol=3)
    sub = B[(B.landscape != "unstructured") & (B.noise == 1.0)]; labs = list(sub.landscape); w = 0.2
    for k, (col, lab, c) in enumerate((("gap enum", "score every code once, take the best", COLORS["gray"]), ("gap N=256", "elite search, N = 256", COLORS["orange"]), ("gap N=64", "elite search, N = 64", COLORS["green"]))): ax[1].bar(np.arange(len(labs)) + (k - 1) * w, sub[col], w, color=c, label=lab)
    ax[1].set(xticks=range(len(labs)), xticklabels=labs, ylabel="how far the chosen code is from the best\n(standard deviations of the true score)", title="(b) with noisy scores (noise sd = 1)", ylim=(0, 1.15)); ax[1].legend(frameon=False, fontsize=7, loc="upper right"); plt.tight_layout(); return fig
def curse_sim(tau, sig, G=30, M=1024, reps=300, seed=0):
    rg = np.random.default_rng(seed); app, tru = [], []
    for _ in range(reps):
        g = tau * rg.normal(size=(G, M)); e = sig * rg.normal(size=(G, M)); s = np.argmax(g + e, axis=1); app.append((g + e)[np.arange(G), s].mean()); tru.append(g[np.arange(G), s].mean())
    return float(np.mean(app)), float(np.mean(tru))
def fig_curse():
    rg = np.random.default_rng(2); M = 1024; tau, sig = 0.5, 1.0; g = tau * rg.normal(size=M); obs = g + sig * rg.normal(size=M); s = int(np.argmax(obs))
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); ax[0].scatter(g, obs, s=6, color=COLORS["gray"], alpha=0.6); ax[0].scatter(g[s], obs[s], s=70, color=COLORS["red"], zorder=3); ax[0].axhline(obs[s], color=COLORS["red"], ls=":", lw=0.8); ax[0].axvline(g[s], color=COLORS["blue"], ls=":", lw=0.8)
    ax[0].annotate(f"chosen code: measured {obs[s]:.2f},\ntrue effect only {g[s]:.2f}", (g[s], obs[s]), (-1.45, 2.6), fontsize=8, arrowprops=dict(arrowstyle="->", color=COLORS["red"])); ax[0].set(xlabel="true effect of the code", ylabel="measured effect (true + noise)", title="(a) picking the best of 1024 noisy measurements")
    shares = [0, 0.25, 0.5, 0.75, 1.0]; tot = 0.0374 / 3.244; A, T = [], []
    for sh in shares: a, t = curse_sim(tot * math.sqrt(1 - sh), tot * math.sqrt(sh), reps=120); A.append(a); T.append(t)
    ax[1].plot(shares, A, "o-", color=COLORS["blue"], label="apparent gain of the best code"); ax[1].plot(shares, T, "s-", color=COLORS["red"], label="true gain of that code"); ax[1].set(xlabel="share of the code-to-code differences that is measurement noise", ylabel="gain over the default code", title="(b) the same apparent gain, different true gains"); ax[1].legend(frameon=False, fontsize=7); plt.tight_layout(); return fig, shares, A, T
def fig_reported():
    fig, ax = plt.subplots(figsize=(6.2, 3.5)); v = [0.1914, 0.2233, 0.2288]; labs = ["default code", "found by the\nelite search", "best of all\n1024 codes"]; ax.bar(range(3), v, color=[COLORS["gray"], COLORS["orange"], COLORS["red"]], width=0.6); ax.set_ylim(0.15, 0.24)
    for i, x in enumerate(v): ax.text(i, x + 0.001, f"{x:.4f}", ha="center", fontsize=9)
    ax.annotate("", xy=(2.45, 0.2288), xytext=(2.45, 0.1914), arrowprops=dict(arrowstyle="<->")); ax.text(2.5, 0.21, "available gain\n0.0374", fontsize=8, va="center"); ax.text(1, 0.162, "the search recovers\n0.0319 of it (85%)", ha="center", fontsize=8); ax.set(xticks=range(3), xticklabels=labs, ylabel="score on 30 held-out genes (higher = better)", xlim=(-0.5, 3.4)); plt.tight_layout(); return fig
def distinct_draws(n=30, M=1024, S=50000, seed=0):
    rg = np.random.default_rng(seed); return np.array([len(set(rg.integers(0, M, n))) for _ in range(S)])
def fig_codes():
    nd = distinct_draws(); fig, ax = plt.subplots(figsize=(6.4, 3.5)); vals, cnt = np.unique(nd, return_counts=True); ax.bar(vals, cnt / len(nd), color=COLORS["blue"], alpha=0.85); ax.axvline(26, color=COLORS["red"], lw=2); ax.text(25.7, ax.get_ylim()[1] * 0.8, "observed: 26", color=COLORS["red"], ha="right", fontsize=9)
    ax.set(xlabel="number of different codes among 30 genes", ylabel="probability", title=f"if the 30 best codes were random picks of 1024\nP(26 or fewer) = {np.mean(nd <= 26):.4f}", xlim=(20, 31)); plt.tight_layout(); return fig
def fit_logreg(X, y, lam, it=300):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(it): p = sigm(X @ w + b); w -= 0.5 * (X.T @ (p - y) / len(y) + lam * w); b -= 0.5 * (p - y).mean()
    return w, b
def amortize(s, p=200, ntr=268, nte=30, d=10, lam=0.05, seed=0):
    rg = np.random.default_rng(seed); X = rg.normal(size=(ntr + nte, p)); W = np.zeros((p, d)); W[:5] = rg.normal(size=(5, d)) * s; codes = (X @ W + rg.normal(size=(ntr + nte, d)) > 0).astype(int); P = np.zeros((nte, d))
    for j in range(d): w, b = fit_logreg(X[:ntr], codes[:ntr, j], lam); P[:, j] = sigm(X[ntr:] @ w + b)
    pred = (P > 0.5).astype(int); return (pred == codes[ntr:]).mean(), np.mean((pred == codes[ntr:]).all(1))
def amortize_table(strengths=(0.0, 0.5, 1.0, 2.0, 4.0), reps=20):
    return pd.DataFrame([(s, *np.mean([amortize(s, seed=k) for k in range(reps)], 0)) for s in strengths], columns=["strength", "bit_acc", "exact"])
def fig_actor(A):
    fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.plot(A.strength, A.bit_acc, "o-", color=COLORS["green"], label="fraction of bits predicted correctly"); ax.plot(A.strength, A.exact, "s-", color=COLORS["red"], label="fraction of whole codes predicted correctly"); ax.axhline(0.5, color="gray", ls=":", lw=0.9); ax.text(4.0, 0.51, "chance for a bit", ha="right", fontsize=8, color="gray")
    ax.set(xlabel="how strongly the code depends on the gene features\n(0 = not at all)", ylabel="on 30 unseen genes", ylim=(-0.02, 0.9)); ax.legend(frameon=False, fontsize=7, loc="center right"); plt.tight_layout(); return fig

In [ ]:
import math, time, importlib.util, os, sys, re, json, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
np.set_printoptions(precision=4, suppress=True, linewidth=120)

## 0. Cells as points, and the scale of the score
A single cell has an *expression profile*: the number of RNA molecules of each of about $18{,}000$ genes. So a cell is a point in a space with $18{,}000$ dimensions (we draw two of them). Turning one gene down moves the cloud of cells; a virtual cell predicts where the cloud goes.
The score compares the predicted cloud with the measured one through six numbers; each is rescaled as $(u-b)/(r-b)$ with $b$ the value of a **baseline** (guess the average response; $0$) and $r$ the value of a **replicate** (repeat the experiment; $1$); the overall score is their average.
**Stop and predict:** the study's scores are $0.1914$, $0.2233$ and $0.2288$. Are they close to $0$, to $1$, or in between?

In [ ]:
fig_cells(); plt.show(); fig_scale(); plt.show()
def scaled_score(u, b, r): return (u - b) / (r - b)
print("raw value at the baseline ->", scaled_score(0.40, 0.40, 0.90), "| at the replicate ->", scaled_score(0.90, 0.40, 0.90), "| halfway ->", scaled_score(0.65, 0.40, 0.90))
README_ROWS = {"entry 1": ([0.7341, 0.1364, 0.1566, -0.0138, 0.1288, 0.0031], 0.19088), "entry 2": ([0.7335, 0.1579, 0.1559, -0.0139, 0.1274, 0.0031], 0.19396), "entry 3": ([0.7408, 0.1955, 0.1276, -0.0111, 0.1323, 0.0037], 0.19813), "entry 4": ([0.7350, 0.1908, 0.1528, -0.0123, 0.1362, 0.0033], 0.20097)}
print(pd.DataFrame([(k, round(float(np.mean(v)), 5), o) for k, (v, o) in README_ROWS.items()], columns=["leaderboard entry", "average of its six scaled scores", "overall score printed"]).to_string(index=False))
print("the overall score is the plain average of the six scaled scores; one of them (about +0.73) is much larger than the other five")
assert all(abs(np.mean(v) - o) < 5e-5 for v, o in README_ROWS.values())

## 1. The elite search by hand: a six-bit worked example
We keep, for each bit, the probability that it is $1$, like ten biased coins; at the start every coin is fair. One *cycle*: **(1)** sample $N$ codes from the coins; **(2)** score each; **(3)** keep the best $K$ (the *elite*) and count the ones in each column; **(4)** nudge each coin toward the elite: $q_i=\frac{c_i+1/2}{K+1}$, then $\theta_i\leftarrow(1-\alpha)\theta_i+\alpha\log\frac{q_i}{1-q_i}$ where $\theta=\log\frac{p}{1-p}$ is the *log-odds*.
Here the hidden target is $101101$, the score is the fraction of matching bits (the search never sees the target, only the scores), $N=12$, $K=3$, $\alpha=0.4$.
**Stop and predict:** after one cycle, which coins tilt most strongly, and in which direction?

In [ ]:
w = worked_example()                                  # seed 11: twelve codes from fair coins
tab = pd.DataFrame(w["B"], columns=[f"b{i + 1}" for i in range(6)], index=np.arange(1, 13)); tab["matches"] = (w["B"] == TARGET6).sum(1); tab["score"] = w["r"].round(3); tab["elite"] = ["yes" if i in w["el"] else "" for i in range(12)]
print("target:", "".join(map(str, TARGET6))); print(tab.to_string())
print("\nthe 3 best codes are numbers", [int(i) for i in w["el"] + 1], "-> ones per column among them: c =", [int(c) for c in w["c"]])
print("q = (c + 1/2)/(K + 1)        =", w["q"].round(3)); print("log-odds of q                =", w["lam"].round(3)); print("new theta = 0.4 * log-odds   =", w["th_new"].round(3)); print("new probabilities p          =", w["p_new"].round(3))
print("target bits                  =", TARGET6)
print("every coin has tilted toward the target bit, most where the three elite codes agreed (bits 1, 2, 5, 6), least where they split (bits 3, 4)")
assert list(w["el"] + 1) == [11, 5, 12] and list(w["c"]) == [3, 0, 2, 2, 0, 3] and np.allclose(w["p_new"], [0.685, 0.315, 0.551, 0.551, 0.315, 0.685], atol=1e-3)

In [ ]:
fig_cem_heat(); plt.show()
print("eight bits, 60 samples and 6 elites per cycle: each row of (a) is one cycle; the coins settle on the target within a few cycles")

## 2. Why half a pseudocount: the ceiling of the search
If the raw frequency $c_i/K$ were used, a bit that is $1$ in every elite code would get probability exactly $1$ and could never change again. Adding half an imaginary success and half an imaginary failure, $q=(c+\frac12)/(K+1)$, keeps $q$ strictly inside $(0,1)$. The largest it can be is $q_{\max}=\frac{K+1/2}{K+1}$, and the coin can never be more certain than that. We watch the search with two numbers: the **entropy** $H$ (average uncertainty of the coins in bits: $1$ fair, $0$ certain) and the **polarization** $P$ (average distance from fair: $0$ fair, $1$ certain).
**Stop and predict:** with $K=102$ elites, what are the largest probability, $H$ and $P$ that the search can reach?

In [ ]:
rows = []
for K in (5, 10, 26, 102):
    q = (K + 0.5) / (K + 1); rows.append((K, round(q, 5), round(entropy_bits([q]), 4), round(polarization([q]), 4), round(q ** 24, 3)))
print(pd.DataFrame(rows, columns=["K elites", "largest probability (K+1/2)/(K+1)", "entropy H there", "polarization P there", "chance that all 24 bits of a sample are right"]).to_string(index=False))
K = 102; pmax = (K + 0.5) / (K + 1); fig_pseudocount(); plt.show()
print(f"with K = {K}: p_max = {pmax:.5f}, H = {entropy_bits([pmax]):.4f}, P = {polarization([pmax]):.4f}")
assert abs(entropy_bits([pmax]) - 0.0443) < 5e-5 and abs(polarization([pmax]) - 0.9903) < 5e-5

## 3. A test with a known answer: 24 bits
Before trusting a search on a problem whose best answer is unknown, test it on one whose answer is known. Hidden 24-bit target; the score of a code is $1-$(fraction of wrong bits); each cycle samples $N=1024$ codes and keeps the best $K=102$ (the settings of the study). After each cycle we draw *fresh* codes from the current coins and record their average score (fresh, so that we measure the policy and not the elite samples that built it). The black squares are the values that the study reports for its own 24-bit test.
**Stop and predict:** after how many cycles will the most likely code equal the target, and where will the curves of $H$ and $P$ flatten?

In [ ]:
h, opt = run_synthetic(); fig_synthetic(h); plt.show()
ours = h.set_index("cycle").loc[REPORTED.index, ["mean", "H", "P"]]; print(pd.concat({"reported by the study": REPORTED, "our run": ours}, axis=1).round(4).to_string())
first = int(h.cycle[h.acc >= 1.0].iloc[0]); dev = float(np.abs(ours[["H", "P"]].values[1:] - REPORTED[["H", "P"]].values[1:]).max())
print(f"\nthe most likely code equals the target from cycle {first} (the study: within four); H and P stay within {dev:.3f} of the reported values from cycle 4 on and flatten at the ceiling 0.0443 and 0.9903")
print(f"at the ceiling a fresh 24-bit sample is exactly the target with probability {pmax ** 24:.2f}, so the average fresh score stays a little below its maximum: {h['mean'].iloc[-1]:.4f} here")
print("the first average score differs (0.57 here, 0.53 reported): the study's score falls faster with each wrong bit than our simple one; the update rule fixes the shape of the H and P curves, which match")
assert first <= 4 and dev < 0.04 and abs(h.H.iloc[-1] - 0.0443) < 1e-3

## 4. Judging a search: the work it does
Because there are only $1024$ codes we can **enumerate** (score all of them): this gives the highest score reachable by any search within the model and is the reference. But a cycle of $N=1024$ samples draws as many codes as the whole space, and the study reports about $20$ cycles. What counts is the number of **different** codes scored. On ten-bit problems whose true scores we know, we compare enumeration (score every code once, take the best) with the elite search at three sample sizes, with and without noise on the scores. A problem is *additive* when each bit acts independently, *pairwise* when bits interact in pairs, *unstructured* when every code is unrelated to its neighbours.
**Stop and predict:** with $N=1024$, what fraction of the $1024$ codes does the search score in $20$ cycles?

In [ ]:
t0 = time.perf_counter(); BUD = budget_table(); print(BUD.round(2).to_string(index=False)); print(f"[{time.perf_counter() - t0:.0f} s]"); fig_budget(BUD); plt.show()
print("N = 1024 scores 840 to 1024 of the 1024 codes in 20 cycles, about the cost of enumeration; N = 64 scores about 250 to 375 on the additive and pairwise problems and loses little")
print("with noisy scores, 'score every code once and take the best' lands 0.7 to 0.8 standard deviations away from the true best, because it picks lucky errors (next section); the elite search averages over many evaluations")
print("on the unstructured problem no bit-by-bit search can work (gap about 1 sd); there enumeration is the only reliable method")
a_ = BUD[BUD.landscape == "additive"]; assert (a_["codes N=1024"] > 700).all() and (a_["codes N=64"] < 450).all() and BUD[(BUD.landscape == "additive") & (BUD.noise == 1.0)]["gap enum"].iloc[0] > 0.5

## 5. Judging a search: noise and the winner's curse
If each evaluation of a code is noisy (in the study the predicted counts are rounded to whole numbers at random), the best of $1024$ measured scores is partly *luck*: the code with the highest measurement is, on average, the one with the largest lucky error as well, so a new measurement of it is expected to be lower. For Gaussian effects with spread $\tau$ and noise $\sigma$ the true gain of the chosen code is the fraction $\tau^2/(\tau^2+\sigma^2)$ of the apparent gain. The study's apparent gain of the best code over the default is $0.2288-0.1914=0.0374$; its noise is not reported, so we show what the number would mean for different noise shares.
**Stop and predict:** if every difference between codes were measurement noise, how large would the noise have to be to produce an apparent gain of $0.0374$ by choosing the best of $1024$?

In [ ]:
rg = np.random.default_rng(0); emax = float(np.mean([rg.normal(size=1024).max() for _ in range(4000)])); print(f"the largest of 1024 standard normal numbers is {emax:.3f} on average; so pure noise of sd {0.0374 / emax:.4f} would give an apparent gain of 0.0374")
fig, shares, A_, T_ = fig_curse(); plt.show()
print(pd.DataFrame({"share of the differences that is noise": shares, "apparent gain": np.round(A_, 4), "true gain of the chosen code": np.round(T_, 4)}).to_string(index=False))
print("the same apparent gain is compatible with a true gain anywhere from 0.0374 to 0; the cure is to re-score the chosen code with new random numbers (a fresh score is not affected by the selection)")
assert abs(A_[0] - 0.0374) < 0.002 and T_[-1] < 0.001 and T_[0] > 0.035

## 6. The result on real cells (as reported)
The study applies the search to $30$ held-out genes of one human cell line, using the measured cells of each gene to compute its score. It reports the **average score over the 30 genes** for the default setting, for the code found by the search and for the best of all $1024$ codes.
**Stop and predict:** what fraction of the gain available inside the model does the search capture?

In [ ]:
fig_reported(); plt.show()
default, found, best = 0.1914, 0.2233, 0.2288; gain = (found - default) / (best - default)
print(f"available gain {best - default:.4f}; captured by the search {found - default:.4f} = {gain:.1%}; left to enumeration {best - found:.4f}")
print("also reported: about 20 cycles; for 11 of the 30 genes the search ends within 0.001 of the best code; no weight of the network changed")
print("caveats: (i) the scores are on the cells used to find the codes (in-sample); (ii) the best code is a maximum of 1024 noisy scores (Section 5); (iii) the scores are small on the scale 0 to 1 of the first figure")
assert abs(gain - 0.853) < 0.002

## 7. Can the search be skipped? The actor, and a birthday problem
The search needs measured cells. An **actor** is a second model trained to predict the winning code from a description of the gene (in the study, $5{,}120$ numbers per gene), using genes that *were* searched ($268$ of them) as training examples; for a new gene it outputs a code without any search. Learning a shortcut from the results of an expensive procedure is called *amortization*. The study's four actors scored between $-0.1773$ and $+0.1889$ on the $30$ held-out genes, none above the default $0.1914$. What can such a failure mean? We build the pipeline on synthetic data where we control how much the code depends on the features.
**Stop and predict:** if the code depends strongly on five of two hundred features, what fraction of bits will a linear actor get right on unseen genes, and how often the whole ten-bit code?

In [ ]:
t0 = time.perf_counter(); AMZ = amortize_table(); print(AMZ.round(3).to_string(index=False)); print(f"chance: 0.500 per bit, {1 / 1024:.4f} for a whole code [{time.perf_counter() - t0:.0f} s]"); fig_actor(AMZ); plt.show()
print("with no dependence the actor is at chance; even with a strong dependence it gets about three bits in four, and a whole code in about one case in ten")
print("so 'no better than a random code' is what an absent relation gives, and also what a weak relation hidden among thousands of features gives: the test cannot tell them apart")
assert AMZ.bit_acc.iloc[0] < 0.53 and AMZ.bit_acc.iloc[-1] > 0.7

**A birthday problem.** The study reports that the $30$ genes needed $26$ different best codes. If every gene's best code were a random pick from $1024$ codes, how many different codes would $30$ genes have, and how unusual is $26$?

In [ ]:
exp_distinct = 1024 * (1 - (1 - 1 / 1024) ** 30); nd = distinct_draws(); print(f"expected number of different codes among 30 random picks from 1024: {exp_distinct:.2f}  (about {30 - exp_distinct:.2f} coincidences)"); print(f"probability of 26 or fewer by chance: {np.mean(nd <= 26):.4f}")
fig_codes(); plt.show()
print("26 different codes is FEWER than chance: some codes are chosen by several genes; the number fits a repertoire of the order of a hundred codes. 'Almost every gene has its own code' is true; 'the codes are unrelated' is not.")
assert abs(exp_distinct - 29.58) < 0.01 and np.mean(nd <= 26) < 0.002

## 8. Your turn
**Exercise 1.** Run Section 3 with $K=10$ and with $K=26$. Predict the ceiling first from $(K+\frac12)/(K+1)$.
**Exercise 2.** Replace the falling step size by a constant $0.9$ (`step=lambda t: 0.9` in `cem`). What changes in the first cycles, and what could go wrong?
**Exercise 3.** Use the raw frequency $c_i/K$ with $K=5$ instead of the pseudocount version and watch a bit freeze.
**Exercise 4.** Give every code in the ten-bit problem an unrelated random score. Why does the elite search fail?
**Exercise 5.** Repeat the winner's-curse simulation with $2000$ codes instead of $1024$. What happens to the expected maximum, and to the apparent gain?

## 9. Self-check

In [ ]:
assert list(w["el"] + 1) == [11, 5, 12] and abs(entropy_bits([pmax]) - 0.0443) < 5e-5 and first <= 4 and abs(gain - 0.853) < 0.002 and np.mean(nd <= 26) < 0.002 and T_[-1] < 0.001
print("All checks passed: you reproduced the numbers in the post.")

# Part II (optional): checks against SciPy and the public model code
> **Read this first.** Part II needs a network connection and, for the last stage, PyTorch. **(1)** SciPy: our probability, log-odds and entropy functions against `scipy.special` and `scipy.stats.entropy`. **(2)** The public code of the Qydra virtual cell (about $200$ MB; shallow clone): its list of the $30$ held-out genes; whether it contains an implementation of the elite search (the study's search scripts were not part of it); and its scorer's scaling $(u-b)/(r-b)$ with the clamps and the plain average of six, compared with our `scaled_score`. **(3)** Its qubit simulator: a single rotation $R_y(\theta)$ gives $\langle Z\rangle=\cos\theta$, and a rotation followed by a CNOT gives two correlated outcomes with probabilities $\cos^2(\theta/2)$ and $\sin^2(\theta/2)$. The trained network and the challenge data are not public, so the experiment on real cells is not rerun. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
try:
    from scipy.special import expit, logit as slogit; x_ = np.linspace(-6, 6, 25); p_ = np.linspace(0.01, 0.99, 25)
    check("our sigmoid and log-odds equal scipy.special.expit and logit", np.allclose(sigm(x_), expit(x_)) and np.allclose(logit(p_), slogit(p_)))
    pv = np.array([0.1, 0.5, 0.9, pmax]); check("our average entropy (bits) equals scipy.stats.entropy of the coins", abs(entropy_bits(pv) - np.mean([stats.entropy([p, 1 - p], base=2) for p in pv])) < 1e-12)
    th_ = np.array([0.0, 1.0, -2.0]); q_ = np.array([0.7, 0.4, 0.99]); a_ = alpha_t(5); check("one smoothed update equals theta + alpha (log-odds(q) - theta)", np.allclose((1 - a_) * th_ + a_ * logit(q_), th_ + a_ * (logit(q_) - th_)))
except Exception as e_: print("scipy stage could not be completed:", type(e_).__name__, e_)

In [ ]:
REPO = "Qydra"
try:
    if not os.path.isdir(REPO): subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/SamuelYueYu/Qydra.git", REPO])
    ho = json.load(open(os.path.join(REPO, "analysis", "holdout_seed0_ho30.json"))); print("held-out genes:", " ".join(ho["holdout"]))
    check("the public held-out list has 30 different genes", len(ho["holdout"]) == 30 and len(set(ho["holdout"])) == 30)
    hits = []
    for root, _, files in os.walk(REPO):
        if ".git" in root: continue
        for f in files:
            if f.endswith((".py", ".md", ".sh", ".json", ".ipynb")):
                t_ = open(os.path.join(root, f), errors="ignore").read().lower()
                if re.search(r"\b(cem|cross-entropy method|elite)\b|bernoulli.{0,40}(search|policy|latent)", t_): hits.append(os.path.join(root, f))
    print("files that mention an elite search or cross-entropy method:", hits if hits else "none"); check("the public code contains no elite-search implementation (it is described in the study)", not hits)
    sys.path.insert(0, os.path.join(REPO, "src")); import metrics_vcc2026 as mv
    names = mv.METRIC_NAMES; raw = dict(zip(names, [0.62, 0.90, 0.55, 0.40, 0.33, 0.48])); base = dict(zip(names, [0.50, 0.60, 0.40, 0.20, 0.10, 0.20])); rep = dict(zip(names, [0.80, 0.30, 0.60, 0.50, 0.40, 0.60]))
    sc = mv.scale_scores(raw, base, rep); mine = [scaled_score(raw[n], base[n], rep[n]) for n in names]; mine[1] = float(np.clip(mine[1], 0, 1)); mine[5] = max(mine[5], -6.0)
    check("the public scorer's scaling equals (u - b)/(r - b) with the expression error limited to [0, 1] and the fold-change error to at least -6", np.allclose([sc[n] for n in names], mine))
    check("the overall score is the plain average of the six scaled scores", abs(sc["context_score"] - np.mean(mine)) < 1e-12)
except Exception as e_: print("repository stage could not be completed:", type(e_).__name__, str(e_)[:300])

In [ ]:
try:
    import torch, qsim
    th = 0.7; st = qsim.zero_state(2, 1, "cpu"); st = qsim.ry(st, 2, 0, torch.tensor(th)); ez = qsim.expval_z(st, 2, 2).detach().cpu().numpy().ravel(); print("RY(0.7) on wire 0: <Z> per wire =", ez.round(4), "| cos(0.7) =", round(math.cos(th), 4))
    check("a single rotation RY(theta) gives <Z> = cos(theta) on the rotated qubit and +1 on the other", abs(max(ez) - 1.0) < 1e-5 and abs(min(ez) - math.cos(th)) < 1e-5)
    s2 = qsim.cnot(st, 2, 0, 1); pr = qsim.probabilities(s2).detach().cpu().numpy().ravel(); print("probabilities after the rotation and a CNOT:", pr.round(4), "| cos^2(theta/2), sin^2(theta/2) =", round(math.cos(th / 2) ** 2, 4), round(math.sin(th / 2) ** 2, 4))
    check("rotation then CNOT gives probabilities cos^2(theta/2) and sin^2(theta/2) on two correlated outcomes", abs(pr.sum() - 1) < 1e-5 and abs(np.sort(pr)[-1] - max(math.cos(th / 2) ** 2, math.sin(th / 2) ** 2)) < 1e-5 and np.sum(pr > 1e-6) == 2)
except Exception as e_: print("simulator stage could not be completed:", type(e_).__name__, str(e_)[:300])
if checks:
    ok_ = [l for l, p_ in checks if p_]; bad_ = [l for l, p_ in checks if not p_]; print(f"\nPart II: {len(ok_)} of {len(checks)} checks passed")
    for l in bad_: print("  not passed:", l)